In [ ]:
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import os
import re

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
BASE_URL = "https://books.toscrape.com/"
FIXED_GBP_TO_INR = 105.50

DB_PATH = "books.db"

print("GBP to INR rate:", FIXED_GBP_TO_INR)
print("Database:", DB_PATH)

GBP to INR rate: 105.5
Database: books.db


In [ ]:
url = "https://books.toscrape.com/"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=15
)

print("Status code:", response.status_code)
print("Website accessible:", response.status_code == 200)

Status code: 200
Website accessible: True


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_books(num_pages=5):
    books = []

    for page in range(1, num_pages + 1):

        if page == 1:
            url = "https://books.toscrape.com/"
        else:
            url = f"https://books.toscrape.com/catalogue/page-{page}.html"

        print(f"Scraping page {page}: {url}")

        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=15
        )

        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        products = soup.select("article.product_pod")

        print(f"Books found: {len(products)}")

        for product in products:


            title_tag = product.select_one("h3 a")
            title = (
                title_tag.get("title", "").strip()
                if title_tag
                else None
            )


            price_tag = product.select_one(".price_color")
            price = (
                price_tag.get_text(strip=True)
                if price_tag
                else None
            )


            availability_tag = product.select_one(".availability")
            availability = (
                availability_tag.get_text(" ", strip=True)
                if availability_tag
                else None
            )

            rating_tag = product.select_one(".star-rating")

            star_rating = None

            if rating_tag:
                classes = rating_tag.get("class", [])

                for word in ["One", "Two", "Three", "Four", "Five"]:
                    if word in classes:
                        star_rating = word
                        break


            book_link = (
                title_tag.get("href")
                if title_tag
                else None
            )

            category = "Unknown"
            if book_link:

                book_url = requests.compat.urljoin(
                    url,
                    book_link
                )

                try:
                    book_response = requests.get(
                        book_url,
                        headers={"User-Agent": "Mozilla/5.0"},
                        timeout=15
                    )

                    book_response.raise_for_status()

                    book_soup = BeautifulSoup(
                        book_response.text,
                        "html.parser"
                    )

                    breadcrumb = book_soup.select(
                        "ul.breadcrumb li a"
                    )


                    if len(breadcrumb) >= 3:
                        category = breadcrumb[-1].get_text(
                            strip=True
                        )

                except requests.RequestException:
                    category = "Unknown"


            books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category
            })

    return pd.DataFrame(books)

In [ ]:
raw_df = scrape_books(num_pages=5)

print("\nScraping completed!")
print("Total books scraped:", len(raw_df))

Scraping page 1: https://books.toscrape.com/
Books found: 20
Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
Books found: 20
Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
Books found: 20
Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
Books found: 20
Scraping page 5: https://books.toscrape.com/catalogue/page-5.html
Books found: 20

Scraping completed!
Total books scraped: 100


In [ ]:
print("Shape:", raw_df.shape)

print("\nColumns:")
print(raw_df.columns.tolist())

print("\nFirst 10 books:")
display(raw_df.head(10))

Shape: (100, 5)

Columns:
['title', 'price', 'star_rating', 'availability', 'category']

First 10 books:


,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History
5,The Requiem Red,Â£22.65,One,In stock,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,Â£33.34,Four,In stock,Business
7,The Coming Woman: A Novel Based on the Life of...,Â£17.93,Three,In stock,Default
8,The Boys in the Boat: Nine Americans and Their...,Â£22.60,Four,In stock,Default
9,The Black Maria,Â£52.15,One,In stock,Poetry


In [ ]:
print("Total books:", len(raw_df))
print("Number of categories:", raw_df["category"].nunique())

print("\nCategories:")
print(raw_df["category"].unique())

print("\nMissing values:")
print(raw_df.isnull().sum())

Total books: 100
Number of categories: 29

Categories:
['Poetry' 'Historical Fiction' 'Fiction' 'Mystery' 'History' 'Young Adult'
 'Business' 'Default' 'Sequential Art' 'Music' 'Science Fiction'
 'Politics' 'Travel' 'Thriller' 'Food and Drink' 'Romance' 'Childrens'
 'Nonfiction' 'Art' 'Spirituality' 'Philosophy' 'New Adult' 'Contemporary'
 'Fantasy' 'Add a comment' 'Science' 'Health' 'Horror' 'Self Help']

Missing values:
title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64


In [ ]:
import re
import pandas as pd

FIXED_GBP_TO_INR = 105.50


def clean_data(df):
    df = df.copy()


    def parse_price(value):
        if pd.isna(value):
            return None

        try:
            match = re.search(r"\d+(?:\.\d+)?", str(value))

            if match:
                return float(match.group())

        except Exception:
            pass

        return None

    df["price_gbp"] = df["price"].apply(parse_price)


    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    df["rating"] = df["star_rating"].map(rating_map)



    def parse_stock(value):
        if pd.isna(value):
            return False

        text = str(value).strip().lower()

        if "out of stock" in text:
            return False

        if "in stock" in text:
            return True

        return False

    df["in_stock"] = df["availability"].apply(parse_stock)


    for column in ["price_gbp", "rating"]:

        if df[column].isna().any():

            median_value = df[column].median()

            df[column] = df[column].fillna(median_value)

            print(
                f"Median imputation applied to {column}: "
                f"{median_value}"
            )


    df["category"] = df["category"].fillna("Unknown")


    df["price_inr"] = (
        df["price_gbp"] * FIXED_GBP_TO_INR
    ).round(2)


    df["price_gbp"] = df["price_gbp"].astype(float)

    df["rating"] = (
        df["rating"]
        .round()
        .astype(int)
    )

    df["in_stock"] = df["in_stock"].astype(bool)

    df["price_inr"] = df["price_inr"].astype(float)


    df = df[
        [
            "title",
            "price_gbp",
            "price_inr",
            "rating",
            "in_stock",
            "category"
        ]
    ]

    return df

In [ ]:
clean_df = clean_data(raw_df)

print("Cleaning completed!")
print("Number of records:", len(clean_df))

Cleaning completed!
Number of records: 100


In [ ]:
display(clean_df.head(10))

,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.74,3,True,Poetry
1,Tipping the Velvet,53.74,5669.57,1,True,Historical Fiction
2,Soumission,50.10,5285.55,1,True,Fiction
3,Sharp Objects,47.82,5045.01,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History
5,The Requiem Red,22.65,2389.57,1,True,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.37,4,True,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,1891.62,3,True,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,2384.30,4,True,Default
9,The Black Maria,52.15,5501.82,1,True,Poetry


In [ ]:
print(clean_df.dtypes)

title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


In [ ]:
print("Minimum rating:", clean_df["rating"].min())
print("Maximum rating:", clean_df["rating"].max())

print("\nRating values:")
print(sorted(clean_df["rating"].unique()))

Minimum rating: 1
Maximum rating: 5

Rating values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [ ]:
expected_price_inr = (
    clean_df["price_gbp"] * 105.50
).round(2)

conversion_check = (
    clean_df["price_inr"] == expected_price_inr
)

print(
    "All INR prices correctly calculated:",
    conversion_check.all()
)

All INR prices correctly calculated: True


In [ ]:
assert len(clean_df) >= 60, \
    "Dataset must contain at least 60 books."

assert clean_df["category"].nunique() >= 3, \
    "Dataset must contain at least 3 categories."

assert clean_df["price_gbp"].notna().all(), \
    "price_gbp contains missing values."

assert clean_df["price_inr"].notna().all(), \
    "price_inr contains missing values."

assert clean_df["rating"].between(1, 5).all(), \
    "Rating must be between 1 and 5."

assert clean_df["in_stock"].dtype == bool, \
    "in_stock must be boolean."

assert conversion_check.all(), \
    "Incorrect GBP to INR conversion."

print("========================================")
print("ALL CLEANING VALIDATIONS PASSED")
print("========================================")
print("Books:", len(clean_df))
print("Categories:", clean_df["category"].nunique())
print("GBP → INR rate: 1 GBP = 105.50 INR")
print("Rating range: 1–5")
print("Conversion correct: True")

ALL CLEANING VALIDATIONS PASSED
Books: 100
Categories: 29
GBP → INR rate: 1 GBP = 105.50 INR
Rating range: 1–5
Conversion correct: True


In [ ]:
import sqlite3
import os

DB_PATH = "books.db"

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,

    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("SQLite database created successfully!")
print("Database file:", DB_PATH)

SQLite database created successfully!
Database file: books.db


In [ ]:
categories = sorted(
    clean_df["category"].unique()
)

for category in categories:
    cursor.execute(
        """
        INSERT INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

conn.commit()

print("Categories inserted:", len(categories))

print("\nCategories:")
for category in categories:
    print("-", category)

Categories inserted: 29

Categories:
- Add a comment
- Art
- Business
- Childrens
- Contemporary
- Default
- Fantasy
- Fiction
- Food and Drink
- Health
- Historical Fiction
- History
- Horror
- Music
- Mystery
- New Adult
- Nonfiction
- Philosophy
- Poetry
- Politics
- Romance
- Science
- Science Fiction
- Self Help
- Sequential Art
- Spirituality
- Thriller
- Travel
- Young Adult


In [ ]:
cursor.execute("""
    SELECT category_id, category_name
    FROM categories
""")

category_map = {
    category_name: category_id
    for category_id, category_name in cursor.fetchall()
}

print("Category mapping:")
print(category_map)

Category mapping:
{'Add a comment': 1, 'Art': 2, 'Business': 3, 'Childrens': 4, 'Contemporary': 5, 'Default': 6, 'Fantasy': 7, 'Fiction': 8, 'Food and Drink': 9, 'Health': 10, 'Historical Fiction': 11, 'History': 12, 'Horror': 13, 'Music': 14, 'Mystery': 15, 'New Adult': 16, 'Nonfiction': 17, 'Philosophy': 18, 'Poetry': 19, 'Politics': 20, 'Romance': 21, 'Science': 22, 'Science Fiction': 23, 'Self Help': 24, 'Sequential Art': 25, 'Spirituality': 26, 'Thriller': 27, 'Travel': 28, 'Young Adult': 29}


In [ ]:
for _, row in clean_df.iterrows():

    cursor.execute(
        """
        INSERT INTO books (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            float(row["price_gbp"]),
            float(row["price_inr"]),
            int(row["rating"]),
            int(row["in_stock"]),
            category_map[row["category"]]
        )
    )

conn.commit()

print("Books inserted successfully!")

Books inserted successfully!


In [ ]:
cursor.execute("SELECT COUNT(*) FROM books")
book_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM categories")
category_count = cursor.fetchone()[0]

print("Books in database:", book_count)
print("Categories in database:", category_count)

Books in database: 100
Categories in database: 29


In [ ]:
cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""")

tables = cursor.fetchall()

print("Tables in database:")

for table in tables:
    print("-", table[0])

Tables in database:
- books
- categories
- sqlite_sequence


In [ ]:
print("CATEGORIES TABLE")

cursor.execute("""
    PRAGMA table_info(categories)
""")

for column in cursor.fetchall():
    print(column)

CATEGORIES TABLE
(0, 'category_id', 'INTEGER', 0, None, 1)
(1, 'category_name', 'TEXT', 1, None, 0)


In [ ]:
print("BOOKS TABLE")

cursor.execute("""
    PRAGMA table_info(books)
""")

for column in cursor.fetchall():
    print(column)

BOOKS TABLE
(0, 'book_id', 'INTEGER', 0, None, 1)
(1, 'title', 'TEXT', 1, None, 0)
(2, 'price_gbp', 'REAL', 1, None, 0)
(3, 'price_inr', 'REAL', 1, None, 0)
(4, 'rating', 'INTEGER', 1, None, 0)
(5, 'in_stock', 'INTEGER', 1, None, 0)
(6, 'category_id', 'INTEGER', 1, None, 0)


In [ ]:
cursor.execute("""
    PRAGMA foreign_key_list(books)
""")

foreign_keys = cursor.fetchall()

print("Foreign keys in books table:")

for fk in foreign_keys:
    print(fk)

Foreign keys in books table:
(0, 0, 'categories', 'category_id', 'category_id', 'NO ACTION', 'NO ACTION', 'NONE')


In [ ]:
test_join = pd.read_sql_query("""
    SELECT
        b.title,
        b.price_gbp,
        b.rating,
        c.category_name
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
    LIMIT 10
""", conn)

display(test_join)

,title,price_gbp,rating,category_name
0,A Light in the Attic,51.77,3,Poetry
1,Tipping the Velvet,53.74,1,Historical Fiction
2,Soumission,50.10,1,Fiction
3,Sharp Objects,47.82,4,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5,History
5,The Requiem Red,22.65,1,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,Default
9,The Black Maria,52.15,1,Poetry


In [ ]:
assert book_count >= 60, \
    "Database must contain at least 60 books."

assert category_count >= 3, \
    "Database must contain at least 3 categories."

assert len(foreign_keys) >= 1, \
    "Books table must have a foreign key."

print("========================================")
print("DATABASE VALIDATION PASSED")
print("========================================")
print(f"Books: {book_count}")
print(f"Categories: {category_count}")
print("Primary keys: Present")
print("Foreign key: Present")
print("Relationship: categories → books")

DATABASE VALIDATION PASSED
Books: 100
Categories: 29
Primary keys: Present
Foreign key: Present
Relationship: categories → books


In [ ]:
queries = {

    # Query 1: SELECT + WHERE
    "query_1_select_where": """
        SELECT
            title,
            price_gbp,
            rating,
            in_stock
        FROM books
        WHERE rating >= 4
    """,

    # Query 2: ORDER BY
    "query_2_order_by": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY price_gbp DESC
    """,

    # Query 3: LIMIT
    "query_3_limit": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY rating DESC
        LIMIT 10
    """,

    # Query 4: DISTINCT
    "query_4_distinct": """
        SELECT DISTINCT
            c.category_name
        FROM categories c
        JOIN books b
            ON c.category_id = b.category_id
        ORDER BY c.category_name
    """,

    # Query 5: BETWEEN
    "query_5_between": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp
    """,

    # Query 6: JOIN
    "query_6_join": """
        SELECT
            b.title,
            b.price_gbp,
            b.price_inr,
            b.rating,
            b.in_stock,
            c.category_name
        FROM books b
        JOIN categories c
            ON b.category_id = c.category_id
        ORDER BY b.rating DESC, b.title
        LIMIT 10
    """
}

print("SQL queries created:", len(queries))

SQL queries created: 6


In [ ]:
query_results = {}

for query_name, query_string in queries.items():

    print("\n" + "=" * 70)
    print(query_name.upper())
    print("=" * 70)

    print("SQL:")
    print(query_string.strip())

    result = pd.read_sql_query(
        query_string,
        conn
    )

    query_results[query_name] = result

    print("\nOUTPUT:")
    display(result)


QUERY_1_SELECT_WHERE
SQL:
SELECT
            title,
            price_gbp,
            rating,
            in_stock
        FROM books
        WHERE rating >= 4

OUTPUT:


,title,price_gbp,rating,in_stock
0,Sharp Objects,47.82,4,1
1,Sapiens: A Brief History of Humankind,54.23,5,1
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4,1
3,The Boys in the Boat: Nine Americans and Their...,22.60,4,1
4,Shakespeare's Sonnets,20.66,4,1
5,Set Me Free,17.46,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5,1
7,Rip it Up and Start Again,35.02,5,1
8,Chase Me (Paris Nights #2),25.27,5,1
9,Black Dust,34.53,5,1



QUERY_2_ORDER_BY
SQL:
SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY price_gbp DESC

OUTPUT:


,title,price_gbp,rating
0,The Death of Humanity: and the Case for Life,58.11,4
1,Slow States of Collapse: Poems,57.31,3
2,Our Band Could Be Your Life: Scenes from the A...,57.25,3
3,The Past Never Ends,56.50,4
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,1
...,...,...,...
95,"Starving Hearts (Triangular Trade Trilogy, #1)",13.99,2
96,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5
97,Princess Between Worlds (Wide-Awake Princess #5),13.34,5
98,In Her Wake,12.84,1



QUERY_3_LIMIT
SQL:
SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY rating DESC
        LIMIT 10

OUTPUT:


...

,category_name
0,Add a comment
1,Art
2,Business
3,Childrens
4,Contemporary
5,Default
6,Fantasy
7,Fiction
8,Food and Drink
9,Health



QUERY_5_BETWEEN
SQL:
SELECT
            title,
            price_gbp,
            rating
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp

OUTPUT:


,title,price_gbp,rating
0,The Inefficiency Assassin: Time Management Tac...,20.59,5
1,Shakespeare's Sonnets,20.66,4
2,In the Country We Love: My Family Divided,22.00,4
3,America's Cradle of Quarterbacks: Western Penn...,22.50,3
4,The Boys in the Boat: Nine Americans and Their...,22.60,4
5,The Requiem Red,22.65,1
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,5
7,The Elephant Tree,23.82,5
8,Olio,23.88,1
9,The Mindfulness and Acceptance Workbook for An...,23.89,4



QUERY_6_JOIN
SQL:
SELECT
            b.title,
            b.price_gbp,
            b.price_inr,
            b.rating,
            b.in_stock,
            c.category_name
        FROM books b
        JOIN categories c
            ON b.category_id = c.category_id
        ORDER BY b.rating DESC, b.title
        LIMIT 10

OUTPUT:


,title,price_gbp,price_inr,rating,in_stock,category_name
0,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,1,Nonfiction
1,Black Dust,34.53,3642.92,5,1,Romance
2,Chase Me (Paris Nights #2),25.27,2665.98,5,1,Romance
3,Join,35.67,3763.19,5,1,Science Fiction
4,Princess Between Worlds (Wide-Awake Princess #5),13.34,1407.37,5,1,Fantasy
5,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,1435.86,5,1,Sequential Art
6,Private Paris (Private #10),47.61,5022.85,5,1,Fiction
7,Rip it Up and Start Again,35.02,3694.61,5,1,Music
8,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1,History
9,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,1,Sequential Art


In [ ]:
output_dir = "outputs"

os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(
    output_dir,
    "query_outputs.txt"
)

with open(output_file, "w", encoding="utf-8") as f:

    for query_name, query_string in queries.items():

        f.write("=" * 80 + "\n")
        f.write(query_name.upper() + "\n")
        f.write("=" * 80 + "\n\n")

        f.write("SQL QUERY:\n")
        f.write(query_string.strip() + "\n\n")

        result = query_results[query_name]

        f.write("OUTPUT:\n")
        f.write(result.to_string(index=False))
        f.write("\n\n")

print("Saved:", output_file)

Saved: outputs/query_outputs.txt


In [ ]:
for query_name, result in query_results.items():

    filename = os.path.join(
        output_dir,
        f"{query_name}.csv"
    )

    result.to_csv(
        filename,
        index=False
    )

print("All query result CSV files saved.")

All query result CSV files saved.


In [ ]:
all_sql = " ".join(
    query.upper()
    for query in queries.values()
)

required_sql_features = {
    "SELECT": "SELECT" in all_sql,
    "WHERE": "WHERE" in all_sql,
    "ORDER BY": "ORDER BY" in all_sql,
    "LIMIT": "LIMIT" in all_sql,
    "DISTINCT": "DISTINCT" in all_sql,
    "BETWEEN": "BETWEEN" in all_sql,
    "JOIN": "JOIN" in all_sql
}

print("SQL requirement validation:")
print()

for feature, present in required_sql_features.items():
    print(f"{feature:10} : {present}")

assert all(required_sql_features.values()), \
    "One or more required SQL features are missing."

print("\nALL REQUIRED SQL FEATURES ARE PRESENT!")

SQL requirement validation:

SELECT     : True
WHERE      : True
ORDER BY   : True
LIMIT      : True
DISTINCT   : True
BETWEEN    : True
JOIN       : True

ALL REQUIRED SQL FEATURES ARE PRESENT!


In [ ]:
read_sql_result_1 = pd.read_sql(
    queries["query_1_select_where"],
    conn
)

read_sql_result_2 = pd.read_sql(
    queries["query_6_join"],
    conn
)

print("First pd.read_sql() result:")
display(read_sql_result_1.head(10))

print("\nSecond pd.read_sql() result:")
display(read_sql_result_2.head(10))

First pd.read_sql() result:


,title,price_gbp,rating,in_stock
0,Sharp Objects,47.82,4,1
1,Sapiens: A Brief History of Humankind,54.23,5,1
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4,1
3,The Boys in the Boat: Nine Americans and Their...,22.60,4,1
4,Shakespeare's Sonnets,20.66,4,1
5,Set Me Free,17.46,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5,1
7,Rip it Up and Start Again,35.02,5,1
8,Chase Me (Paris Nights #2),25.27,5,1
9,Black Dust,34.53,5,1



Second pd.read_sql() result:


,title,price_gbp,price_inr,rating,in_stock,category_name
0,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,1,Nonfiction
1,Black Dust,34.53,3642.92,5,1,Romance
2,Chase Me (Paris Nights #2),25.27,2665.98,5,1,Romance
3,Join,35.67,3763.19,5,1,Science Fiction
4,Princess Between Worlds (Wide-Awake Princess #5),13.34,1407.37,5,1,Fantasy
5,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,1435.86,5,1,Sequential Art
6,Private Paris (Private #10),47.61,5022.85,5,1,Fiction
7,Rip it Up and Start Again,35.02,3694.61,5,1,Music
8,Sapiens: A Brief History of Humankind,54.23,5721.26,5,1,History
9,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,1,Sequential Art


In [ ]:
read_sql_result_1.to_csv(
    "outputs/read_sql_select_where.csv",
    index=False
)

read_sql_result_2.to_csv(
    "outputs/read_sql_join.csv",
    index=False
)

print("pd.read_sql() outputs saved successfully.")

pd.read_sql() outputs saved successfully.


In [ ]:
assert len(queries) >= 5

assert "SELECT" in all_sql
assert "WHERE" in all_sql
assert "ORDER BY" in all_sql
assert "LIMIT" in all_sql
assert "DISTINCT" in all_sql
assert "BETWEEN" in all_sql
assert "JOIN" in all_sql

assert isinstance(
    read_sql_result_1,
    pd.DataFrame
)

assert isinstance(
    read_sql_result_2,
    pd.DataFrame
)

print("========================================")
print("SQL VALIDATION PASSED")
print("========================================")
print("Number of SQL queries:", len(queries))
print("SELECT + WHERE: PASS")
print("ORDER BY: PASS")
print("LIMIT: PASS")
print("DISTINCT: PASS")
print("BETWEEN: PASS")
print("JOIN: PASS")
print("pd.read_sql(): PASS")
print("========================================")

SQL VALIDATION PASSED
Number of SQL queries: 6
SELECT + WHERE: PASS
ORDER BY: PASS
LIMIT: PASS
DISTINCT: PASS
BETWEEN: PASS
JOIN: PASS
pd.read_sql(): PASS


In [ ]:
# Get categories from SQLite into a pandas DataFrame
categories_df = pd.read_sql(
    """
    SELECT category_id, category_name
    FROM categories
    """,
    conn
)

display(categories_df)

,category_id,category_name
0,1,Add a comment
1,2,Art
2,3,Business
3,4,Childrens
4,5,Contemporary
5,6,Default
6,7,Fantasy
7,8,Fiction
8,9,Food and Drink
9,10,Health


In [ ]:
category_id_map = dict(
    zip(
        categories_df["category_name"],
        categories_df["category_id"]
    )
)

books_df = clean_df.copy()

books_df["category_id"] = books_df["category"].map(
    category_id_map
)

display(books_df.head())

,title,price_gbp,price_inr,rating,in_stock,category,category_id
0,A Light in the Attic,51.77,5461.74,3,True,Poetry,19
1,Tipping the Velvet,53.74,5669.57,1,True,Historical Fiction,11
2,Soumission,50.10,5285.55,1,True,Fiction,8
3,Sharp Objects,47.82,5045.01,4,True,Mystery,15
4,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History,12


In [ ]:
merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

merge_result = merge_result[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

merge_result = merge_result.sort_values(
    by=["rating", "title"],
    ascending=[False, True]
).head(10).reset_index(drop=True)

display(merge_result)

,title,price_gbp,price_inr,rating,in_stock,category_name
0,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,True,Nonfiction
1,Black Dust,34.53,3642.92,5,True,Romance
2,Chase Me (Paris Nights #2),25.27,2665.98,5,True,Romance
3,Join,35.67,3763.19,5,True,Science Fiction
4,Princess Between Worlds (Wide-Awake Princess #5),13.34,1407.37,5,True,Fantasy
5,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,1435.86,5,True,Sequential Art
6,Private Paris (Private #10),47.61,5022.85,5,True,Fiction
7,Rip it Up and Start Again,35.02,3694.61,5,True,Music
8,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History
9,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,True,Sequential Art


In [ ]:
# Get the SQL JOIN result again
sql_join_result = pd.read_sql(
    queries["query_6_join"],
    conn
)

# Reproduce the same result using pandas merge
merge_result = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

# Select the same columns
merge_result = merge_result[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_name"
    ]
]

# Apply the same sorting and LIMIT as SQL
merge_result = (
    merge_result
    .sort_values(
        by=["rating", "title"],
        ascending=[False, True]
    )
    .head(10)
    .reset_index(drop=True)
)

# ------------------------------------------------
# Normalize data types so both results are equal
# ------------------------------------------------

# SQLite returns 0/1 for boolean values.
# Convert SQL result to actual boolean.
sql_join_result["in_stock"] = (
    sql_join_result["in_stock"].astype(bool)
)

# Make pandas result boolean too.
merge_result["in_stock"] = (
    merge_result["in_stock"].astype(bool)
)

# Ensure numeric columns have consistent types.
sql_join_result["price_gbp"] = (
    sql_join_result["price_gbp"].astype(float).round(2)
)

merge_result["price_gbp"] = (
    merge_result["price_gbp"].astype(float).round(2)
)

sql_join_result["price_inr"] = (
    sql_join_result["price_inr"].astype(float).round(2)
)

merge_result["price_inr"] = (
    merge_result["price_inr"].astype(float).round(2)
)

sql_join_result["rating"] = (
    sql_join_result["rating"].astype(int)
)

merge_result["rating"] = (
    merge_result["rating"].astype(int)
)

print("SQL JOIN result:")
display(sql_join_result)

print("\nPandas pd.merge() result:")
display(merge_result)

SQL JOIN result:


,title,price_gbp,price_inr,rating,in_stock,category_name
0,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,True,Nonfiction
1,Black Dust,34.53,3642.92,5,True,Romance
2,Chase Me (Paris Nights #2),25.27,2665.98,5,True,Romance
3,Join,35.67,3763.19,5,True,Science Fiction
4,Princess Between Worlds (Wide-Awake Princess #5),13.34,1407.37,5,True,Fantasy
5,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,1435.86,5,True,Sequential Art
6,Private Paris (Private #10),47.61,5022.85,5,True,Fiction
7,Rip it Up and Start Again,35.02,3694.61,5,True,Music
8,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History
9,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,True,Sequential Art



Pandas pd.merge() result:


,title,price_gbp,price_inr,rating,in_stock,category_name
0,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,2438.10,5,True,Nonfiction
1,Black Dust,34.53,3642.92,5,True,Romance
2,Chase Me (Paris Nights #2),25.27,2665.98,5,True,Romance
3,Join,35.67,3763.19,5,True,Science Fiction
4,Princess Between Worlds (Wide-Awake Princess #5),13.34,1407.37,5,True,Fantasy
5,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,1435.86,5,True,Sequential Art
6,Private Paris (Private #10),47.61,5022.85,5,True,Fiction
7,Rip it Up and Start Again,35.02,3694.61,5,True,Music
8,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History
9,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5516.60,5,True,Sequential Art


In [ ]:
merge_result = merge_result[
    sql_join_result.columns
]

sql_join_result = sql_join_result.reset_index(drop=True)
merge_result = merge_result.reset_index(drop=True)


results_match = sql_join_result.equals(
    merge_result
)

print("SQL JOIN and pd.merge() results match:")
print(results_match)

SQL JOIN and pd.merge() results match:
True


In [ ]:
comparison_match = sql_join_result.equals(
    merge_result
)

assert comparison_match, \
    "SQL JOIN and pandas merge results do not match."

print("=" * 60)
print("SQL JOIN = PANDAS MERGE")
print("=" * 60)
print("Equivalent output: TRUE")
print("=" * 60)

SQL JOIN = PANDAS MERGE
Equivalent output: TRUE


In [ ]:
pipeline_code = r'''
import os
import re
import sqlite3
import requests
import pandas as pd
from bs4 import BeautifulSoup



BASE_URL = "https://books.toscrape.com/"
FIXED_GBP_TO_INR = 105.50

DB_PATH = "books.db"
OUTPUT_DIR = "outputs"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}



def scrape_books(num_pages=5):

    books = []

    for page in range(1, num_pages + 1):

        if page == 1:
            url = BASE_URL
        else:
            url = (
                f"{BASE_URL}"
                f"catalogue/page-{page}.html"
            )

        print(f"Scraping page {page}: {url}")

        response = requests.get(
            url,
            headers=HEADERS,
            timeout=15
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        products = soup.select(
            "article.product_pod"
        )

        print(
            f"Books found on page {page}: "
            f"{len(products)}"
        )

        for product in products:

            title_tag = product.select_one(
                "h3 a"
            )

            title = (
                title_tag.get("title", "").strip()
                if title_tag
                else None
            )

            price_tag = product.select_one(
                ".price_color"
            )

            price = (
                price_tag.get_text(strip=True)
                if price_tag
                else None
            )

            availability_tag = product.select_one(
                ".availability"
            )

            availability = (
                availability_tag.get_text(
                    " ",
                    strip=True
                )
                if availability_tag
                else None
            )

            rating_tag = product.select_one(
                ".star-rating"
            )

            star_rating = None

            if rating_tag:

                classes = rating_tag.get(
                    "class",
                    []
                )

                for word in [
                    "One",
                    "Two",
                    "Three",
                    "Four",
                    "Five"
                ]:

                    if word in classes:
                        star_rating = word
                        break

            category = "Unknown"

            book_link = (
                title_tag.get("href")
                if title_tag
                else None
            )

            if book_link:

                book_url = requests.compat.urljoin(
                    url,
                    book_link
                )

                try:

                    book_response = requests.get(
                        book_url,
                        headers=HEADERS,
                        timeout=15
                    )

                    book_response.raise_for_status()

                    book_soup = BeautifulSoup(
                        book_response.text,
                        "html.parser"
                    )

                    breadcrumb = book_soup.select(
                        "ul.breadcrumb li a"
                    )

                    if len(breadcrumb) >= 3:

                        category = (
                            breadcrumb[-1]
                            .get_text(strip=True)
                        )

                except requests.RequestException:

                    category = "Unknown"

            books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category
            })

    return pd.DataFrame(books)



def clean_data(df):

    df = df.copy()

    def parse_price(value):

        if pd.isna(value):
            return None

        try:

            match = re.search(
                r"\d+(?:\.\d+)?",
                str(value)
            )

            if match:
                return float(match.group())

        except Exception:
            pass

        return None

    df["price_gbp"] = df[
        "price"
    ].apply(parse_price)

    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }

    df["rating"] = df[
        "star_rating"
    ].map(rating_map)

    def parse_stock(value):

        if pd.isna(value):
            return False

        text = str(value).strip().lower()

        if "out of stock" in text:
            return False

        if "in stock" in text:
            return True

        return False

    df["in_stock"] = df[
        "availability"
    ].apply(parse_stock)

    # Median imputation for numeric fields
    for column in [
        "price_gbp",
        "rating"
    ]:

        if df[column].isna().any():

            median_value = df[
                column
            ].median()

            df[column] = df[
                column
            ].fillna(median_value)

    df["category"] = df[
        "category"
    ].fillna("Unknown")

    # Fixed project-defined conversion
    df["price_inr"] = (
        df["price_gbp"]
        * FIXED_GBP_TO_INR
    ).round(2)

    df["price_gbp"] = df[
        "price_gbp"
    ].astype(float)

    df["rating"] = (
        df["rating"]
        .round()
        .astype(int)
    )

    df["in_stock"] = df[
        "in_stock"
    ].astype(bool)

    df["price_inr"] = df[
        "price_inr"
    ].astype(float)

    return df[
        [
            "title",
            "price_gbp",
            "price_inr",
            "rating",
            "in_stock",
            "category"
        ]
    ]



def create_database(clean_df):

    conn = sqlite3.connect(DB_PATH)

    conn.execute(
        "PRAGMA foreign_keys = ON"
    )

    cursor = conn.cursor()

    cursor.execute(
        "DROP TABLE IF EXISTS books"
    )

    cursor.execute(
        "DROP TABLE IF EXISTS categories"
    )

    cursor.execute("""
        CREATE TABLE categories (
            category_id INTEGER PRIMARY KEY AUTOINCREMENT,
            category_name TEXT UNIQUE NOT NULL
        )
    """)

    cursor.execute("""
        CREATE TABLE books (
            book_id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            price_gbp REAL NOT NULL,
            price_inr REAL NOT NULL,
            rating INTEGER NOT NULL,
            in_stock INTEGER NOT NULL,
            category_id INTEGER NOT NULL,
            FOREIGN KEY (category_id)
                REFERENCES categories(category_id)
        )
    """)

    categories = sorted(
        clean_df["category"].unique()
    )

    for category in categories:

        cursor.execute(
            """
            INSERT INTO categories (
                category_name
            )
            VALUES (?)
            """,
            (category,)
        )

    cursor.execute("""
        SELECT category_id, category_name
        FROM categories
    """)

    category_map = {
        name: category_id
        for category_id, name
        in cursor.fetchall()
    }

    for _, row in clean_df.iterrows():

        cursor.execute(
            """
            INSERT INTO books (
                title,
                price_gbp,
                price_inr,
                rating,
                in_stock,
                category_id
            )
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                row["title"],
                float(row["price_gbp"]),
                float(row["price_inr"]),
                int(row["rating"]),
                int(row["in_stock"]),
                category_map[
                    row["category"]
                ]
            )
        )

    conn.commit()

    return conn




def run_queries(conn):

    queries = {

        "query_1_select_where": """
            SELECT
                title,
                price_gbp,
                rating,
                in_stock
            FROM books
            WHERE rating >= 4
        """,

        "query_2_order_by": """
            SELECT
                title,
                price_gbp,
                rating
            FROM books
            ORDER BY price_gbp DESC
        """,

        "query_3_limit": """
            SELECT
                title,
                price_gbp,
                rating
            FROM books
            ORDER BY rating DESC
            LIMIT 10
        """,

        "query_4_distinct": """
            SELECT DISTINCT
                c.category_name
            FROM categories c
            JOIN books b
                ON c.category_id = b.category_id
            ORDER BY c.category_name
        """,

        "query_5_between": """
            SELECT
                title,
                price_gbp,
                rating
            FROM books
            WHERE price_gbp BETWEEN 20 AND 40
            ORDER BY price_gbp
        """,

        "query_6_join": """
            SELECT
                b.title,
                b.price_gbp,
                b.price_inr,
                b.rating,
                b.in_stock,
                c.category_name
            FROM books b
            JOIN categories c
                ON b.category_id = c.category_id
            ORDER BY
                b.rating DESC,
                b.title
            LIMIT 10
        """
    }

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    results = {}

    output_file = os.path.join(
        OUTPUT_DIR,
        "query_outputs.txt"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:

        for name, query in queries.items():

            result = pd.read_sql(
                query,
                conn
            )

            results[name] = result

            f.write("=" * 80 + "\n")
            f.write(name.upper() + "\n")
            f.write("=" * 80 + "\n\n")

            f.write("SQL QUERY:\n")
            f.write(query.strip() + "\n\n")

            f.write("OUTPUT:\n")
            f.write(
                result.to_string(
                    index=False
                )
            )

            f.write("\n\n")

            result.to_csv(
                os.path.join(
                    OUTPUT_DIR,
                    f"{name}.csv"
                ),
                index=False
            )

    return queries, results




def compare_sql_and_merge(
    clean_df,
    conn,
    sql_join_result
):

    categories_df = pd.read_sql(
        """
        SELECT
            category_id,
            category_name
        FROM categories
        """,
        conn
    )

    category_id_map = dict(
        zip(
            categories_df["category_name"],
            categories_df["category_id"]
        )
    )

    books_df = clean_df.copy()

    books_df["category_id"] = (
        books_df["category"]
        .map(category_id_map)
    )

    merge_result = pd.merge(
        books_df,
        categories_df,
        on="category_id",
        how="inner"
    )

    merge_result = merge_result[
        [
            "title",
            "price_gbp",
            "price_inr",
            "rating",
            "in_stock",
            "category_name"
        ]
    ]

    merge_result = (
        merge_result
        .sort_values(
            by=["rating", "title"],
            ascending=[False, True]
        )
        .head(10)
        .reset_index(drop=True)
    )

    sql_join_result = (
        sql_join_result
        .copy()
        .reset_index(drop=True)
    )

    sql_join_result["in_stock"] = (
        sql_join_result["in_stock"]
        .astype(bool)
    )

    merge_result["in_stock"] = (
        merge_result["in_stock"]
        .astype(bool)
    )

    for column in [
        "price_gbp",
        "price_inr"
    ]:

        sql_join_result[column] = (
            sql_join_result[column]
            .astype(float)
            .round(2)
        )

        merge_result[column] = (
            merge_result[column]
            .astype(float)
            .round(2)
        )

    sql_join_result["rating"] = (
        sql_join_result["rating"]
        .astype(int)
    )

    merge_result["rating"] = (
        merge_result["rating"]
        .astype(int)
    )

    merge_result = merge_result[
        sql_join_result.columns
    ]

    merge_result = (
        merge_result
        .reset_index(drop=True)
    )

    results_match = (
        sql_join_result.equals(
            merge_result
        )
    )

    sql_join_result.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "sql_join_result.csv"
        ),
        index=False
    )

    merge_result.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "merge_results.csv"
        ),
        index=False
    )

    with open(
        os.path.join(
            OUTPUT_DIR,
            "sql_vs_merge_comparison.txt"
        ),
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            "SQL JOIN RESULT\n"
        )

        f.write(
            sql_join_result.to_string(
                index=False
            )
        )

        f.write(
            "\n\nPANDAS pd.merge() RESULT\n"
        )

        f.write(
            merge_result.to_string(
                index=False
            )
        )

        f.write(
            f"\n\nResults equivalent: "
            f"{results_match}\n"
        )

    return (
        categories_df,
        books_df,
        merge_result,
        results_match
    )




def main():

    print("=" * 70)
    print("ZEpto DATA PIPELINE")
    print("=" * 70)

    # Scrape
    raw_df = scrape_books(
        num_pages=5
    )

    print(
        f"\nTotal books scraped: "
        f"{len(raw_df)}"
    )

    # Validate scraping
    assert len(raw_df) >= 60
    assert raw_df["category"].nunique() >= 3

    # Clean
    clean_df = clean_data(
        raw_df
    )

    print(
        f"Cleaned books: "
        f"{len(clean_df)}"
    )

    # Validate conversion
    expected_price_inr = (
        clean_df["price_gbp"]
        * FIXED_GBP_TO_INR
    ).round(2)

    assert (
        clean_df["price_inr"]
        == expected_price_inr
    ).all()

    # Database
    conn = create_database(
        clean_df
    )

    print(
        f"Database created: "
        f"{DB_PATH}"
    )

    # SQL
    queries, results = run_queries(
        conn
    )

    print(
        f"SQL queries executed: "
        f"{len(queries)}"
    )

    # pd.read_sql requirement
    read_sql_result_1 = pd.read_sql(
        queries[
            "query_1_select_where"
        ],
        conn
    )

    read_sql_result_2 = pd.read_sql(
        queries[
            "query_6_join"
        ],
        conn
    )

    read_sql_result_1.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "read_sql_select_where.csv"
        ),
        index=False
    )

    read_sql_result_2.to_csv(
        os.path.join(
            OUTPUT_DIR,
            "read_sql_join.csv"
        ),
        index=False
    )

    # SQL JOIN vs pandas merge
    (
        categories_df,
        books_df,
        merge_result,
        results_match
    ) = compare_sql_and_merge(
        clean_df,
        conn,
        read_sql_result_2
    )

    assert results_match

    # Final validation
    assert len(clean_df) >= 60
    assert clean_df["category"].nunique() >= 3
    assert clean_df["rating"].between(
        1, 5
    ).all()

    assert clean_df["in_stock"].dtype == bool

    assert (
        clean_df["price_inr"]
        == (
            clean_df["price_gbp"]
            * FIXED_GBP_TO_INR
        ).round(2)
    ).all()

    print("\n" + "=" * 70)
    print("DATA PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 70)
    print(
        f"Books: {len(clean_df)}"
    )
    print(
        f"Categories: "
        f"{clean_df['category'].nunique()}"
    )
    print(
        "GBP → INR: "
        "1 GBP = 105.50 INR"
    )
    print(
        "SQL JOIN = pandas merge: "
        f"{results_match}"
    )
    print(
        f"Database: {DB_PATH}"
    )
    print(
        f"Outputs: {OUTPUT_DIR}/"
    )
    print("=" * 70)

    conn.close()


if __name__ == "__main__":
    main()
'''

with open("pipeline.py", "w", encoding="utf-8") as f:
    f.write(pipeline_code)

print("pipeline.py created successfully!")

pipeline.py created successfully!


In [ ]:
queries_code = r'''
import sqlite3
import pandas as pd


DB_PATH = "books.db"


queries = {

    "query_1_select_where": """
        SELECT
            title,
            price_gbp,
            rating,
            in_stock
        FROM books
        WHERE rating >= 4
    """,

    "query_2_order_by": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY price_gbp DESC
    """,

    "query_3_limit": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY rating DESC
        LIMIT 10
    """,

    "query_4_distinct": """
        SELECT DISTINCT
            c.category_name
        FROM categories c
        JOIN books b
            ON c.category_id = b.category_id
        ORDER BY c.category_name
    """,

    "query_5_between": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp
    """,

    "query_6_join": """
        SELECT
            b.title,
            b.price_gbp,
            b.price_inr,
            b.rating,
            b.in_stock,
            c.category_name
        FROM books b
        JOIN categories c
            ON b.category_id = c.category_id
        ORDER BY
            b.rating DESC,
            b.title
        LIMIT 10
    """
}


def main():

    conn = sqlite3.connect(DB_PATH)

    print("=" * 70)
    print("SQL QUERY RESULTS")
    print("=" * 70)

    for name, query in queries.items():

        print("\n" + "-" * 70)
        print(name)
        print("-" * 70)

        print(query.strip())

        result = pd.read_sql(
            query,
            conn
        )

        print("\nOUTPUT:")
        print(result.to_string(index=False))

    conn.close()


if __name__ == "__main__":
    main()
'''

with open("queries.py", "w", encoding="utf-8") as f:
    f.write(queries_code)

print("queries.py created successfully!")

queries.py created successfully!


In [ ]:
requirements = """requests
beautifulsoup4
pandas
"""

with open(
    "requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [ ]:
import os

print("Files created:")

for file in [
    "pipeline.py",
    "queries.py",
    "requirements.txt"
]:
    print(
        f"{file}: "
        f"{os.path.exists(file)}"
    )

Files created:
pipeline.py: True
queries.py: True
requirements.txt: True


In [ ]:
lines = [
"# Module 1 - Data Pipeline",
"",
"## Overview",
"",
"This module implements a complete data pipeline for scraping, cleaning, enriching, storing, and querying catalogue data.",
"",
"Data source: books.toscrape.com",
"",
"The pipeline follows:",
"",
"Scrape -> Clean -> Convert -> Store -> Query -> Compare",
"",
"## 1. Data Source",
"",
"The project uses books.toscrape.com, a public scraping-practice website.",
"",
"The pipeline scrapes the first 5 paginated catalogue pages and collects at least 60 books.",
"",
"Fields collected:",
"- title",
"- price",
"- star_rating",
"- availability",
"- category",
"",
"## 2. Technologies Used",
"",
"- Python",
"- requests",
"- BeautifulSoup",
"- pandas",
"- SQLite",
"- sqlite3",
"",
"## 3. Installation",
"",
"Install the required packages:",
"",
"pip install -r requirements.txt",
"",
"SQLite is provided through Python's built-in sqlite3 module.",
"",
"## 4. Running the Pipeline",
"",
"Run the complete pipeline using:",
"",
"python pipeline.py",
"",
"The script automatically scrapes, cleans, converts, stores, queries, and validates the data.",
"",
"## 5. Cleaning and Parsing Decisions",
"",
"### Price",
"",
"The currency symbol is removed and the price is converted to a float.",
"",
"Example: GBP 51.77 becomes 51.77.",
"",
"The resulting column is price_gbp.",
"",
"### Star Rating",
"",
"The website provides ratings as text: One, Two, Three, Four, Five.",
"",
"These are converted to integers:",
"",
"One -> 1",
"Two -> 2",
"Three -> 3",
"Four -> 4",
"Five -> 5",
"",
"The resulting column is rating.",
"",
"### Availability",
"",
"Availability text is converted into a boolean value.",
"",
"In stock -> True",
"Out of stock -> False",
"",
"The resulting column is in_stock.",
"",
"SQLite stores boolean values as 1 and 0.",
"",
"### Missing Numeric Values",
"",
"If a numeric field fails to parse, median imputation is used.",
"",
"This preserves the maximum amount of scraped catalogue data instead of unnecessarily dropping rows.",
"",
"### Missing Category",
"",
"If a category cannot be obtained, it is assigned the value Unknown.",
"",
"## 6. Currency Conversion",
"",
"The required fixed project-defined conversion rate is:",
"",
"1 GBP = 105.50 INR",
"",
"This is an artificial fixed baseline specified by the project.",
"",
"No live currency API is required or used.",
"",
"The calculation is:",
"",
"price_inr = price_gbp * 105.50",
"",
"The result is rounded to two decimal places.",
"",
"## 7. Database Design",
"",
"The project uses SQLite with two normalized tables.",
"",
"### categories",
"",
"categories(category_id INTEGER PRIMARY KEY, category_name TEXT UNIQUE)",
"",
"### books",
"",
"books(book_id INTEGER PRIMARY KEY, title TEXT, price_gbp REAL, price_inr REAL, rating INTEGER, in_stock INTEGER, category_id INTEGER, FOREIGN KEY(category_id) REFERENCES categories(category_id))",
"",
"The categories table stores unique category names.",
"",
"The books table stores book information and references categories using category_id.",
"",
"## 8. SQL Queries",
"",
"Six SQL queries are included.",
"",
"They collectively demonstrate:",
"- SELECT",
"- WHERE",
"- ORDER BY",
"- LIMIT",
"- DISTINCT",
"- BETWEEN",
"- JOIN",
"",
"Query 1 uses SELECT and WHERE to find highly rated books.",
"Query 2 uses ORDER BY to sort books by price.",
"Query 3 uses LIMIT to return the top 10 records.",
"Query 4 uses DISTINCT to list unique categories.",
"Query 5 uses BETWEEN to filter books by price range.",
"Query 6 uses JOIN to combine books and category information.",
"",
"## 9. Pandas SQL Reading",
"",
"At least two SQL query results are read into pandas DataFrames using pd.read_sql().",
"",
"The results are saved in the outputs directory.",
"",
"## 10. SQL JOIN vs pandas.merge()",
"",
"The SQL JOIN result is independently reproduced using pd.merge().",
"",
"The SQL JOIN and pandas merge results are normalized for compatible data types and compared using DataFrame.equals().",
"",
"The final comparison confirms that both approaches produce equivalent output.",
"",
"The comparison is saved in outputs/sql_vs_merge_comparison.txt.",
"",
"## 11. Output Files",
"",
"The outputs directory contains:",
"",
"- query_1_select_where.csv",
"- query_2_order_by.csv",
"- query_3_limit.csv",
"- query_4_distinct.csv",
"- query_5_between.csv",
"- query_6_join.csv",
"- query_outputs.txt",
"- read_sql_select_where.csv",
"- read_sql_join.csv",
"- sql_join_result.csv",
"- merge_results.csv",
"- sql_vs_merge_comparison.txt",
"",
"## 12. Main Files",
"",
"data_pipeline/",
"",
"    pipeline.py",
"    queries.py",
"    requirements.txt",
"    README.md",
"    books.db",
"    outputs/",
"",
"pipeline.py contains the complete end-to-end pipeline.",
"",
"queries.py contains the SQL queries.",
"",
"requirements.txt contains the required Python packages.",
"",
"books.db is the SQLite database generated by the pipeline.",
"",
"outputs contains SQL query results and the SQL-versus-pandas comparison.",
"",
"## 13. Reproducibility",
"",
"The SQLite database can be recreated from scratch by running:",
"",
"python pipeline.py",
"",
"No manual copy-pasting of scraped data is required.",
"",
"## 14. Validation",
"",
"The pipeline validates that:",
"",
"- at least 60 books are available",
"- at least 3 categories are available",
"- ratings are integers from 1 to 5",
"- in_stock is boolean in pandas",
"- INR prices use the required fixed conversion rate",
"- the SQLite database contains the required primary and foreign key relationship",
"- all required SQL clauses are represented",
"- at least two results are read using pd.read_sql()",
"- the SQL JOIN and pandas pd.merge() results are equivalent",
"",
"## 15. Conclusion",
"",
"This module demonstrates a complete raw-to-relational data pipeline:",
"",
"Public Catalogue",
"-> Web Scraping",
"-> Data Cleaning",
"-> Type Conversion",
"-> GBP to INR Enrichment",
"-> Normalized SQLite Database",
"-> SQL Queries",
"-> Pandas Analysis",
"-> SQL JOIN vs pd.merge Validation",
"",
"The pipeline is reproducible and can be regenerated from scratch using the provided Python script."
]

readme_content = "\n".join(lines)

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md created successfully!")
print("Number of characters:", len(readme_content))

README.md created successfully!
Number of characters: 5380


In [ ]:
import os

print("README exists:", os.path.exists("README.md"))

if os.path.exists("README.md"):
    with open("README.md", "r", encoding="utf-8") as f:
        content = f.read()

    print("README size:", len(content), "characters")
    print("\nFirst 15 lines:")
    print("\n".join(content.splitlines()[:15]))

README exists: True
README size: 5380 characters

First 15 lines:
# Module 1 - Data Pipeline

## Overview

This module implements a complete data pipeline for scraping, cleaning, enriching, storing, and querying catalogue data.

Data source: books.toscrape.com

The pipeline follows:

Scrape -> Clean -> Convert -> Store -> Query -> Compare

## 1. Data Source

The project uses books.toscrape.com, a public scraping-practice website.


In [ ]:
import os
import shutil

os.makedirs("data_pipeline", exist_ok=True)
for file in [
    "pipeline.py",
    "queries.py",
    "requirements.txt",
    "README.md",
    "books.db"
]:
    source = file
    destination = os.path.join("data_pipeline", file)

    if os.path.exists(source):
        if os.path.exists(destination):
            os.remove(destination)

        shutil.move(source, destination)

if os.path.exists("outputs"):
    destination = os.path.join(
        "data_pipeline",
        "outputs"
    )

    if os.path.exists(destination):
        shutil.rmtree(destination)

    shutil.move(
        "outputs",
        destination
    )

print("data_pipeline folder prepared successfully!")

data_pipeline folder prepared successfully!


In [ ]:
import os

print("FINAL DATA PIPELINE STRUCTURE")
print("=" * 50)

for root, dirs, files in os.walk("data_pipeline"):

    level = root.replace(
        "data_pipeline",
        ""
    ).count(os.sep)

    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for file in files:
        print(indent + "    " + file)

FINAL DATA PIPELINE STRUCTURE
data_pipeline/
    books.db
    requirements.txt
    pipeline.py
    README.md
    queries.py
    outputs/
        read_sql_join.csv
        query_2_order_by.csv
        query_outputs.txt
        query_4_distinct.csv
        query_6_join.csv
        query_5_between.csv
        read_sql_select_where.csv
        query_1_select_where.csv
        query_3_limit.csv


In [ ]:
!pip install -q requests beautifulsoup4 pandas

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "data_pipeline/pipeline.py"
    ],
    capture_output=True,
    text=True
)

print("========== PROGRAM OUTPUT ==========")
print(result.stdout)

print("\n========== ERRORS ==========")
print(result.stderr)

print("\n========== EXIT CODE ==========")
print(result.returncode)

========== PROGRAM OUTPUT ==========
ZEpto DATA PIPELINE
Scraping page 1: https://books.toscrape.com/
Books found on page 1: 20
Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
Books found on page 2: 20
Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
Books found on page 3: 20
Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
Books found on page 4: 20
Scraping page 5: https://books.toscrape.com/catalogue/page-5.html
Books found on page 5: 20

Total books scraped: 100
Cleaned books: 100
Database created: books.db
SQL queries executed: 6

DATA PIPELINE COMPLETED SUCCESSFULLY
Books: 100
Categories: 29
GBP → INR: 1 GBP = 105.50 INR
SQL JOIN = pandas merge: True
Database: books.db
Outputs: outputs/


========== ERRORS ==========


========== EXIT CODE ==========
0


In [ ]:
import os
import shutil

if os.path.exists("books.db"):
    destination = "data_pipeline/books.db"

    if os.path.exists(destination):
        os.remove(destination)

    shutil.move(
        "books.db",
        destination
    )

if os.path.exists("outputs"):
    destination = "data_pipeline/outputs"

    if os.path.exists(destination):
        shutil.rmtree(destination)

    shutil.move(
        "outputs",
        destination
    )

print("Database and outputs moved into data_pipeline successfully!")

Database and outputs moved into data_pipeline successfully!


In [ ]:
import os

print("FINAL DATA_PIPELINE STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk("data_pipeline"):

    level = root.replace(
        "data_pipeline",
        ""
    ).count(os.sep)

    indent = "    " * level

    print(indent + os.path.basename(root) + "/")

    for file in files:
        print(indent + "    " + file)

FINAL DATA_PIPELINE STRUCTURE
data_pipeline/
    books.db
    requirements.txt
    pipeline.py
    README.md
    queries.py
    outputs/
        sql_join_result.csv
        read_sql_join.csv
        query_2_order_by.csv
        query_outputs.txt
        sql_vs_merge_comparison.txt
        query_4_distinct.csv
        query_6_join.csv
        query_5_between.csv
        read_sql_select_where.csv
        merge_results.csv
        query_1_select_where.csv
        query_3_limit.csv


In [ ]:
import shutil
import os

# Create a ZIP file containing data_pipeline
zip_path = shutil.make_archive(
    "data_pipeline",
    "zip",
    ".",
    "data_pipeline"
)

print("ZIP created:")
print(zip_path)

print("File exists:", os.path.exists(zip_path))

ZIP created:
/content/data_pipeline.zip
File exists: True


In [ ]:
from google.colab import files

files.download("data_pipeline.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>